<a href="https://colab.research.google.com/github/zpsheldon/meg-neural-decoding/blob/main/speech-detection-training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports



In [1]:
# Install additional depdendencies
%pip install -q lightning torchmetrics scikit-learn plotly ipywidgets pnpl

# Set up base path for dataset and related files (base_path is assumed to be set in the cells below!)
base_path = "./libribrain"
try:
    import google.colab  # This module is only available in Colab.
    in_colab = True
    base_path = "/content"  # This is the folder displayed in the Colab sidebar
except ImportError:
    in_colab = False

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.9/168.9 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 94.3 MB/s eta 0:00:00


In [2]:
import pnpl
from pnpl.datasets import LibriBrainSpeech
from torch.utils.data import DataLoader
import pandas as pd
import random
import torch
import numpy as np
import torch.nn as nn
import platform
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import lightning as L
import torchmetrics
from lightning.pytorch.callbacks import Callback
from torchmetrics.functional import recall
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score, roc_curve, auc, confusion_matrix, precision_recall_curve, jaccard_score
from matplotlib.colors import ListedColormap
import os
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger
from lightning.pytorch.callbacks import EarlyStopping
import optuna
from optuna.integration import PyTorchLightningPruningCallback


/opt/miniconda3/envs/pytorch_env/lib/python3.13/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


ModuleNotFoundError: 
Could not find `optuna-integration` for `pytorch_lightning`.
Please run `pip install optuna-integration[pytorch_lightning]`.

## Load data

In [ ]:
toy_training = True

In [4]:
num_books = 7
num_chapters = [9, 12, 12, 12, 15, 14, 14]

In [5]:
# Conditionally set num_workers to avoid multiprocessing issues (try increasing if performance is problematic)
num_workers = 2 if in_colab else 0

# For training, we'll use all sessions from Sherlock 1 to Sherlock 5
train_run_keys = []
if toy_training:
  train_run_keys = [("0",str(i),f"Sherlock1","1") for i in range(1,3)]
else:
  train_run_keys = [("0",str(i),f"Sherlock{j}","1") for j in range(1,6) for i in range(1, num_chapters[j-1])]
train_data = LibriBrainSpeech(
  data_path=f"{base_path}/data/",
  include_run_keys = train_run_keys,
  tmin=0.0,
  tmax=0.8,
  preload_files = True
)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=num_workers)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


(…)-0_ses-1_task-Sherlock1_run-1_events.tsv: 0.00B [00:00, ?B/s]

(…)-0_ses-2_task-Sherlock1_run-1_events.tsv: 0.00B [00:00, ?B/s]

Sherlock1/derivatives/serialised/sub-0_s(…):   0%|          | 0.00/429M [00:00<?, ?B/s]

Sherlock1/derivatives/serialised/sub-0_s(…):   0%|          | 0.00/342M [00:00<?, ?B/s]

Done!
calculated stats for:  ('0', '1', 'Sherlock1', '1')
calculated stats for:  ('0', '2', 'Sherlock1', '1')


In [7]:
# For validation, we'll use all sessions from Sherlock 6
val_run_keys = []
if toy_training:
  val_run_keys = [("0","3","Sherlock1","1")]
else:
  val_run_keys = [("0",i,"Sherlock6","1") for i in range(1, num_chapters[5])]
val_data = LibriBrainSpeech(
  data_path=f"{base_path}/data/",
  include_run_keys=val_run_keys,
  standardize=True,
  tmin=0.0,
  tmax=0.8,
  preload_files = True
)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=num_workers)

(…)-0_ses-3_task-Sherlock1_run-1_events.tsv: 0.00B [00:00, ?B/s]

Sherlock1/derivatives/serialised/sub-0_s(…):   0%|          | 0.00/461M [00:00<?, ?B/s]

Done!
calculated stats for:  ('0', '3', 'Sherlock1', '1')


In [8]:
# For testing, we'll use all sessions from Sherlock 7
test_run_keys = []
if toy_training:
  test_run_keys = [("0","4","Sherlock1","1")]
else:
  test_run_keys = [("0",i,"Sherlock7","1") for i in range(1, num_chapters[6])]
test_data = LibriBrainSpeech(
  data_path=f"{base_path}/data/",
  include_run_keys=test_run_keys,
  standardize=True,
  tmin=0.0,
  tmax=0.8,
  preload_files = True
)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=num_workers)

(…)-0_ses-4_task-Sherlock1_run-1_events.tsv: 0.00B [00:00, ?B/s]

Sherlock1/derivatives/serialised/sub-0_s(…):   0%|          | 0.00/300M [00:00<?, ?B/s]

Done!
calculated stats for:  ('0', '4', 'Sherlock1', '1')


In [9]:
print("Number of training samples:", len(train_data))
print("Number of validation samples:", len(val_data))
print("Number of test samples:", len(test_data))

Number of training samples: 3097
Number of validation samples: 1849
Number of test samples: 1195


## Filter data

In [10]:
# These are the sensors we identified as being particularly useful
SENSORS_SPEECH_MASK = [18, 20, 22, 23, 45, 120, 138, 140, 142, 143, 145,
                       146, 147, 149, 175, 176, 177, 179, 180, 198, 271, 272, 275]

class FilteredDataset(torch.utils.data.Dataset):
    """
    Parameters:
        dataset: LibriBrain dataset.
        limit_samples (int, optional): If provided, limits the length of the dataset to this
                          number of samples.
        speech_silence_only (bool, optional): If True, only includes segments that are either
                          purely speech or purely silence (with additional balancing).
        apply_sensors_speech_mask (bool, optional): If True, applies a fixed sensor mask to the sensor
                          data in each sample.
    """
    def __init__(self,
                 dataset,
                 limit_samples=None,
                 disable=False,
                 apply_sensors_speech_mask=True):
        self.dataset = dataset
        self.limit_samples = limit_samples
        self.apply_sensors_speech_mask = apply_sensors_speech_mask

        # These are the sensors we identified:
        self.sensors_speech_mask = SENSORS_SPEECH_MASK

        self.balanced_indices = list(range(len(dataset.samples)))
        # Shuffle the indices
        self.balanced_indices = random.sample(self.balanced_indices, len(self.balanced_indices))

    def __len__(self):
        """Returns the number of samples in the filtered dataset."""
        if self.limit_samples is not None:
            return self.limit_samples
        return len(self.balanced_indices)

    def __getitem__(self, index):
        # Map index to the original dataset using balanced indices
        original_idx = self.balanced_indices[index]
        if self.apply_sensors_speech_mask:
            sensors = self.dataset[original_idx][0][self.sensors_speech_mask]
        else:
            sensors = self.dataset[original_idx][0][:]
        label_from_the_middle_idx = self.dataset[original_idx][1].shape[0] // 2
        return [sensors, self.dataset[original_idx][1][label_from_the_middle_idx]]


# Conditionally set num_workers to avoid multiprocessing issues (try increasing if performance is problematic)
num_workers = 2 if in_colab else 0

print("Filtered dataset:")
train_data_filtered = FilteredDataset(train_data)
train_loader_filtered = DataLoader(train_data_filtered, batch_size=32, shuffle=True, num_workers=num_workers)
print(f"Train data contains {len(train_data_filtered)} samples")

val_data_filtered = FilteredDataset(val_data)
val_loader_filtered = DataLoader(val_data_filtered, batch_size=32, shuffle=False, num_workers=num_workers)
print(f"Validation data contains {len(val_data_filtered)} samples")

test_data_filtered = FilteredDataset(test_data)
test_loader_filtered = DataLoader(test_data_filtered, batch_size=32, shuffle=False, num_workers=num_workers)
print(f"Test data contains {len(test_data_filtered)} samples\n")

# Let's look at the first batch:
first_batch = next(iter(train_loader_filtered))
inputs, labels = first_batch
print("Batch input shape:", inputs.shape)
print("Batch label shape:", labels.shape)

first_input = inputs[0]
first_label = labels[0]
print("\nSingle sample input shape:", first_input.shape)
print("Single sample label is just a single value now!")
print("\nFirst sample input:", first_input)
print("First sample label:", first_label)

Filtered dataset:
Train data contains 3097 samples
Validation data contains 1849 samples
Test data contains 1195 samples

Batch input shape: torch.Size([32, 23, 200])
Batch label shape: torch.Size([32])

Single sample input shape: torch.Size([23, 200])
Single sample label is just a single value now!

First sample input: tensor([[ 0.2934,  0.7516,  0.3711,  ...,  1.3550,  1.6096,  1.3907],
        [-1.0541, -0.6286, -0.2811,  ...,  0.6276,  0.6013,  0.5316],
        [-0.7576, -0.7121, -0.4660,  ...,  0.6606,  0.5235,  0.5711],
        ...,
        [-0.1562, -0.3639,  0.2485,  ..., -0.1434, -0.2509, -0.1822],
        [-0.4555, -0.2584, -0.2122,  ...,  0.6966,  0.8290,  1.3763],
        [ 0.1717,  0.8388,  0.2371,  ..., -0.3538,  0.4103,  0.5761]])
First sample label: tensor(1)


## Define model

In [11]:
class BCEWithLogitsLossWithSmoothing(nn.Module):
    def __init__(self, smoothing=0.1, pos_weight = 1.0):
        """
        Binary Cross-Entropy Loss with Deterministic Label Smoothing.

        Parameters:
            smoothing (float): Smoothing factor. Must be between 0 and 1.
            pos_weight (float): Weight for the positive class.
        """
        super().__init__()
        self.smoothing = smoothing
        self.bce_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))

    def forward(self, logits, target):
        target = target.float()  # Ensure target is a float tensor
        target_smoothed = target * (1 - self.smoothing) + self.smoothing * 0.5
        return self.bce_loss(logits, target_smoothed)


In [12]:
class SpeechModel(nn.Module):
    """
    Parameters:
        input_dim (int): Number of channels/features in the input tensor (usually SENSORS_SPEECH_MASK)
        model_dim (int): Dimensionality for the intermediate model representation.
        dropout_rate (float, optional): Dropout probability applied after convolutional and LSTM layers.
        lstm_layers (int, optional): Number of layers in the LSTM module.
        bi_directional (bool, optional): If True, uses a bidirectional LSTM; otherwise, a unidirectional LSTM.
        batch_norm (bool, optional): Indicates whether to use batch normalization.

    """
    def __init__(self, input_dim, model_dim, dropout_rate=0.3, lstm_layers = 1, bi_directional = False, batch_norm=False):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels=input_dim,
            out_channels=model_dim,
            kernel_size=3,
            padding=1,
        )
        self.lstm_layers = lstm_layers
        self.batch_norm = nn.BatchNorm1d(num_features=model_dim) if batch_norm else nn.Identity()
        self.conv_dropout = nn.Dropout(p=dropout_rate)
        self.lstm = nn.LSTM(
            input_size=model_dim,
            hidden_size=model_dim,
            num_layers=self.lstm_layers,
            dropout=dropout_rate,
            batch_first=True,
            bidirectional=bi_directional
        )
        self.lstm_dropout = nn.Dropout(p=dropout_rate)
        self.speech_classifier = nn.Linear(model_dim, 1)

    def forward(self, x):
        x = self.conv(x)
        x = self.batch_norm(x)
        x = self.conv_dropout(x)
        # LSTM expects (batch, seq_len, input_size)
        output, (h_n, c_n) = self.lstm(x.permute(0, 2, 1))
        last_layer_h_n = h_n
        if self.lstm_layers > 1:
            # handle more than one layer
            last_layer_h_n = h_n[-1, :, :]
            last_layer_h_n = last_layer_h_n.unsqueeze(0)
        output = self.lstm_dropout(last_layer_h_n)
        output = output.flatten(start_dim=0, end_dim=1)
        x = self.speech_classifier(output)
        return x

    def predict_proba(self, x):
        logits = self.forward(x)
        probs = torch.sigmoid(logits)
        class_probs = torch.zeros((x.shape[0], 2))
        for i in range(x.shape[0]):
            class_probs[i, 0] = probs[i]
            class_probs[i, 1] = 1-probs[i]
        return class_probs

In [ ]:
class SpeechClassifier(L.LightningModule):
    """
    Parameters:
        input_dim (int): Number of input channels/features. This is passed to the underlying SpeechModel.
        model_dim (int): Dimensionality of the intermediate model representation.
        learning_rate (float, optional): Learning rate for the optimizer.
        weight_decay (float, optional): Weight decay for the optimizer.
        batch_size (int, optional): Batch size used during training and evaluation.
        dropout_rate (float, optional): Dropout probability applied after convolutional and LSTM layers.
        smoothing (float, optional): Label smoothing factor applied in the BCEWithLogits loss.
        pos_weight (float, optional): Weight for the positive class in the BCEWithLogits loss.
        batch_norm (bool, optional): Indicates whether to use batch normalization.
        lstm_layers (int, optional): Number of layers in the LSTM module within the SpeechModel.
        bi_directional (bool, optional): If True, uses a bidirectional LSTM in the SpeechModel; otherwise, uses a unidirectional LSTM.
    """

    def __init__(self, input_dim, model_dim, learning_rate=1e-3, weight_decay=0.01, batch_size=32, dropout_rate=0.3, smoothing=0.1, pos_weight = 1.0 , batch_norm = False, lstm_layers = 1, bi_directional = False):
        super().__init__()
        self.save_hyperparameters()

        self.learning_rate = self.hparams.learning_rate
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.model = SpeechModel(
            input_dim, 
            self.hparams.model_dim, 
            dropout_rate=self.hparams.dropout_rate, 
            lstm_layers=lstm_layers, 
            bi_directional=bi_directional, 
            batch_norm=batch_norm
        )

        self.loss_fn = BCEWithLogitsLossWithSmoothing(smoothing=smoothing, pos_weight = pos_weight)

        self.train_loss = []
        self.val_loss = []

        self.val_step_outputs = []
        self.test_step_outputs = {}


    def forward(self, x):
        return self.model(x)

    def predict_proba(self, x):
        logits = self.forward(x)
        probs = torch.sigmoid(logits)
        class_probs = torch.zeros((x.shape[0], 2))
        for i in range(x.shape[0]):
            class_probs[i, 0] = probs[i]
            class_probs[i, 1] = 1-probs[i]
        return class_probs

    def _shared_eval_step(self, batch, stage):
        x = batch[0]
        y = batch[1] # (batch, seq_len)

        logits = self(x)
        loss = self.loss_fn(logits, y.unsqueeze(1).float())
        probs = torch.sigmoid(logits)
        y_probs = probs.detach().cpu()

        y_true = batch[1].detach().cpu()
        meg = x.detach().cpu()

        self.log(f'{stage}_loss', loss, on_step=False, on_epoch=True, batch_size=self.batch_size)
        return loss


    def training_step(self, batch, batch_idx):
        return self._shared_eval_step(batch, "train")


    def validation_step(self, batch, batch_idx):
        return self._shared_eval_step(batch, "val")


    def test_step(self, batch, batch_idx):
        x = batch[0]
        y = batch[1]  # (batch, seq_len)

        # ugly, taking care of only one label
        if len(y.shape) != 1:
            y = y.flatten(start_dim=0, end_dim=1).view(-1, 1)  # (batch, seq_len) -> (batch * seq_len, 1)
        else:
            y = y.unsqueeze(1)

        logits = self(x)
        loss = self.loss_fn(logits, y.float())
        probs = torch.sigmoid(logits)

        # Append data to the defaultdict
        # Ensure keys exist before appending
        if "y_probs" not in self.test_step_outputs:
            self.test_step_outputs["y_probs"] = []
        if "y_true" not in self.test_step_outputs:
            self.test_step_outputs["y_true"] = []
        if "meg" not in self.test_step_outputs:
            self.test_step_outputs["meg"] = []

        # Append data
        if y.shape[-1] != 1:
            self.test_step_outputs["y_probs"].extend(
                probs.detach().view(x.shape[0], x.shape[-1]).cpu())  # (batch, seq_len)
        else:
            self.test_step_outputs["y_probs"].extend(
                probs.detach().view(x.shape[0], 1).cpu())  # (batch, seq_len)

        self.test_step_outputs["y_true"].extend(batch[1].detach().cpu())  # (batch, seq_len)
        self.test_step_outputs["meg"].extend(x.detach().cpu())  # MEG data (batch, channels, seq_len)

        return self._shared_eval_step(batch, "test")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        return optimizer

## Hyperparameter tuning

In [ ]:
# Set a fixed seed for reproducibility
L.seed_everything(42)

def objective(trial):
    # Set the hyperparameters to optimize
    model_dim = trial.suggest_int('model_dim', 50, 200)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)
    
    model = SpeechClassifier(
        input_dim=len(SENSORS_SPEECH_MASK),
        model_dim=model_dim,
        learning_rate=learning_rate,
        dropout_rate=dropout_rate,
        lstm_layers=2,
        weight_decay=0.01,
        batch_norm=False,
        bi_directional=False
    )
    
    # Early stopping callback
    early_stop_callback = EarlyStopping(
        monitor='val_loss',
        patience=5,
        verbose=False,
        mode='min'
    )
    
    # Optuna pruning callback
    pruning_callback = PyTorchLightningPruningCallback(trial, monitor='val_loss')
    
    # Logger
    logger = TensorBoardLogger(save_dir=os.getcwd(), name=f"optuna_logs/trial_{trial.number}")
    
    # Create trainer
    trainer = L.Trainer(
        max_epochs=10,
        callbacks=[early_stop_callback, pruning_callback],
        logger=logger,
        enable_progress_bar=False,  
        enable_model_summary=False  
    )
    
    # Training the model
    trainer.fit(model, train_loader, val_loader)
    
    # Final validation loss
    return trainer.callback_metrics['val_loss'].item()

In [ ]:
def run_optimization(n_trials=20):
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    study = optuna.create_study(direction='minimize', pruner=pruner)
    study.optimize(objective, n_trials=n_trials)
    
    print("Best trial:")
    trial = study.best_trial
    print(f"  Value: {trial.value}")
    print("  Params: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
        
    return study

## Evaluate best hyperparameter combination

In [ ]:
def test_best_model(study, save:bool=False):
    # Getting the best hyperparameters
    best_params = study.best_trial.params
    
    # Creating the model with the best hyperparameters
    model = SpeechClassifier(
        input_dim=len(SENSORS_SPEECH_MASK),
        model_dim=best_params['model_dim'],
        learning_rate=best_params['learning_rate'],
        dropout_rate=best_params['dropout_rate'],
        lstm_layers=2,
        weight_decay=0.01,
        batch_norm=False,
        bi_directional=False
    )
    
    # Creating trainer instance
    trainer = L.Trainer(max_epochs=10)
    
    # Training the model with the best hyperparameters
    trainer.fit(model, train_loader, val_loader)
    
    # Testing the model with the test data
    results = trainer.test(model, test_loader)
    
    CHECKPOINT_PATH = "speech_detection_tuned.ckpt"
    if save:
        trainer.save_checkpoint(CHECKPOINT_PATH)

    return results, trainer

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    0.5553211569786072     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.5553211569786072}]

## Visualize results

In [ ]:
study = run_optimization(n_trials=5)
  
# Visualize the results
try:
    # Plot optimization history
    optuna.visualization.plot_optimization_history(study)

    # Plot parameter importances
    optuna.visualization.plot_param_importances(study)

    # Plot parallel coordinate plot
    optuna.visualization.plot_parallel_coordinate(study)
except ImportError:
    print("Visualization requires plotly. Install with: pip install plotly")
  
  # Test the best model
results = test_best_model(study, save=(not toy_training))
print(f"Test results with best hyperparameters: {results}")

Ready to compute statistics with 1195 samples.
